In [1]:
import os
import shutil
import numpy as np
from PIL import Image
from collections import defaultdict

In [2]:
# Modify to your own paths

# Dataset base directory
base_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3'

# Official train and test split txt files, available from: https://cvlsegmentation.github.io/driu/downloads.html
train_txt_path = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/RIMONE_train.txt'
test_txt_path = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/RIMONE_test.txt'

# Images original directory
images_original_dir1 = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/Glaucoma and suspects/Stereo Images'
images_original_dir2 = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/Healthy/Stereo Images'

# Masks original directory
masks_original_dir1 = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/Glaucoma and suspects/Average_masks'
masks_original_dir2 = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/RIM-ONE-r3/Healthy/Average_masks'

In [3]:
# Create target directory
train_images_dir = os.path.join(base_dir, 'training', 'images')
train_masks_dir = os.path.join(base_dir, 'training', 'masks')
test_images_dir = os.path.join(base_dir, 'testing', 'images')
test_masks_dir = os.path.join(base_dir, 'testing', 'masks')
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_masks_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_masks_dir, exist_ok=True)

In [4]:
# Get train and test file names
def read_txt(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        filenames = [line.strip() for line in f if line.strip()]
    return list(filenames)  
    
train_filenames = read_txt(train_txt_path)
test_filenames = read_txt(test_txt_path)

In [5]:
# Split images, get the left part, and save to target directory
def process_images(original_dir, target_dir, filenames_list):
    for filename in os.listdir(original_dir):
        name_no_ext = os.path.splitext(filename)[0]            
        if name_no_ext in filenames_list:
            original_path = os.path.join(original_dir, filename)
            target_path = os.path.join(target_dir, filename)

            old_train_image = np.asarray(Image.open(original_path))             
            new_train_image = Image.fromarray(old_train_image[:, 0:1072, :])
            new_train_image.save(target_path, quality=100)

process_images(images_original_dir1, train_images_dir, train_filenames)
process_images(images_original_dir2, train_images_dir, train_filenames)

process_images(images_original_dir1, test_images_dir, test_filenames)
process_images(images_original_dir2, test_images_dir, test_filenames)

In [6]:
# Split masks and get the path lists
def get_masks_names(original_dir, filenames_list, target_list):
    for filename in os.listdir(original_dir):       
        if filename.endswith('Disc-Avg.png'):
            base_name = filename[:-13]
        elif filename.endswith('Cup-Avg.png'):
            base_name = filename[:-12]
        else:
            base_name = 0
                            
        if base_name in filenames_list:
            target_list.append(os.path.join(original_dir, filename))
    return target_list

train_list = []
test_list = []

train_list = get_masks_names(masks_original_dir1, train_filenames, train_list)
train_list = get_masks_names(masks_original_dir2, train_filenames, train_list)

test_list = get_masks_names(masks_original_dir1, test_filenames, test_list)
test_list = get_masks_names(masks_original_dir2, test_filenames, test_list)

In [7]:
# Get the paired OD and OC path lists
def get_paired_masks(paths_list):
    paired_masks = defaultdict(dict)
    for path in paths_list:
        filename = os.path.basename(path)
        parts = filename.split('-')
        patient_id = f'{parts[0]}-{parts[1]}' 
        eye = parts[2]  
        mask_type = parts[3]
        key = f'{patient_id}-{eye}'
            
        if mask_type == 'Disc':
            paired_masks[key]['Disc'] = path
        elif mask_type == 'Cup':
            paired_masks[key]['Cup'] = path
    return paired_masks

train_paired_masks = get_paired_masks(train_list)
test_paired_masks = get_paired_masks(test_list)      

In [8]:
# Modify masks to required form and save to target directory
def process_masks(paired_list, target_dir):
    for key, masks in paired_list.items():
        disc_path = masks['Disc']
        cup_path = masks['Cup']

        disc = np.asarray(Image.open(disc_path)).astype(np.uint16)
        cup = np.asarray(Image.open(cup_path)).astype(np.uint16)

        together = disc + cup
        together[together == 255] = 128
        together[together == 0] = 255
        together[together == 510] = 0
                
        together = Image.fromarray(together[:, 0:1072].astype(np.uint8))
        target_path = os.path.join(target_dir, os.path.basename(disc_path)[:-13]+'.png')
        together.save(target_path, quality=100)

process_masks(train_paired_masks, train_masks_dir)
process_masks(test_paired_masks, test_masks_dir)    